# Data Merging and Lookup Operations with Pandas

This notebook demonstrates advanced data merging techniques and lookup operations using Pandas. We'll work with student, enrollment, and exam score datasets across three comprehensive tasks.

## Task 1: Data Preparation and Missing Value Handling (30 points)

Create the following three DataFrames and handle missing values:
- Students DataFrame: contains student info with some missing values
- Enrollments DataFrame: course enrollments by students
- Scores DataFrame: exam scores for students

In [1]:
import pandas as pd
import numpy as np

# Create Students DataFrame
students_data = {
    'student_id': [101, 102, 103, 104, 105, 106, 107],
    'name': ['Alice', 'Bob', None, 'David', 'Emma', 'Frank', 'Grace'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', None, 'emma@email.com', 'frank@email.com', 'grace@email.com'],
    'city': ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', None, 'Chennai', 'Delhi']
}
students_df = pd.DataFrame(students_data)

# Create Enrollments DataFrame
enrollments_data = {
    'student_id': [101, 102, 103, 105, 108, 109],
    'course_name': ['Python', 'Data Science', 'Python', 'Machine Learning', 'AI', 'Python'],
    'enrollment_date': ['2024-01-15', '2024-01-20', '2024-02-01', '2024-02-10', '2024-02-15', '2024-03-01']
}
enrollments_df = pd.DataFrame(enrollments_data)

# Create Scores DataFrame
scores_data = {
    'student_id': [101, 102, 104, 105, 106],
    'exam_score': [85, 92, 78, 88, 95]
}
scores_df = pd.DataFrame(scores_data)

print("=== ORIGINAL DATA ===\n")
print("Students DataFrame:")
print(students_df)
print("\nEnrollments DataFrame:")
print(enrollments_df)
print("\nScores DataFrame:")
print(scores_df)

=== ORIGINAL DATA ===

Students DataFrame:
   student_id   name              email       city
0         101  Alice    alice@email.com     Mumbai
1         102    Bob      bob@email.com      Delhi
2         103    NaN  charlie@email.com  Bangalore
3         104  David                NaN     Mumbai
4         105   Emma     emma@email.com        NaN
5         106  Frank    frank@email.com    Chennai
6         107  Grace    grace@email.com      Delhi

Enrollments DataFrame:
   student_id       course_name enrollment_date
0         101            Python      2024-01-15
1         102      Data Science      2024-01-20
2         103            Python      2024-02-01
3         105  Machine Learning      2024-02-10
4         108                AI      2024-02-15
5         109            Python      2024-03-01

Scores DataFrame:
   student_id  exam_score
0         101          85
1         102          92
2         104          78
3         105          88
4         106          95


### Task 1.1: Null Value Analysis

In [2]:
# Display null value count and percentage for each column
print("=== NULL VALUE ANALYSIS ===\n")
null_counts = students_df.isnull().sum()
null_percentage = (students_df.isnull().sum() / len(students_df)) * 100

print("Null Value Summary:")
for col in students_df.columns:
    print(f"Column: {col}, Nulls: {null_counts[col]} ({null_percentage[col]:.2f}%)")

=== NULL VALUE ANALYSIS ===

Null Value Summary:
Column: student_id, Nulls: 0 (0.00%)
Column: name, Nulls: 1 (14.29%)
Column: email, Nulls: 1 (14.29%)
Column: city, Nulls: 1 (14.29%)


### Task 1.2: Clean the Students DataFrame

In [3]:
# Create a copy for cleaning
students_clean = students_df.copy()

# Fill missing 'city' values with 'Unknown'
students_clean['city'] = students_clean['city'].fillna('Unknown')

# Drop rows where 'name' is missing
students_clean = students_clean.dropna(subset=['name']).reset_index(drop=True)

print("=== CLEANED STUDENTS DATAFRAME ===\n")
print(f"Cleaned DataFrame ({len(students_clean)} rows):")
print(students_clean)

print("\n--- Summary of Changes ---")
print(f"Original rows: {len(students_df)}")
print(f"After cleaning: {len(students_clean)}")
print(f"Rows removed: {len(students_df) - len(students_clean)}")

=== CLEANED STUDENTS DATAFRAME ===

Cleaned DataFrame (6 rows):
   student_id   name            email     city
0         101  Alice  alice@email.com   Mumbai
1         102    Bob    bob@email.com    Delhi
2         104  David              NaN   Mumbai
3         105   Emma   emma@email.com  Unknown
4         106  Frank  frank@email.com  Chennai
5         107  Grace  grace@email.com    Delhi

--- Summary of Changes ---
Original rows: 7
After cleaning: 6
Rows removed: 1


## Task 2: Multiple Join Operations (40 points)

Perform different join operations (inner, left, right, outer) and analyze the results.

### Task 2.1: Inner Join
Inner join returns only rows where student_id exists in both DataFrames.

In [4]:
# Perform Inner Join: only matching records from both tables
inner_join = pd.merge(students_clean, enrollments_df, on='student_id', how='inner')

print("=== INNER JOIN RESULT ===\n")
print(f"Total rows: {len(inner_join)}")
print(inner_join)

# Analysis
print("\n--- Analysis ---")
students_in_result = inner_join['student_id'].unique()
students_excluded = students_clean[~students_clean['student_id'].isin(students_in_result)]['student_id'].tolist()
print(f"Students in result: {sorted(students_in_result.tolist())}")
print(f"Students excluded: {students_excluded}")
print(f"Reason: Students {students_excluded} don't have enrollment records in the enrollments table.")

=== INNER JOIN RESULT ===

Total rows: 3
   student_id   name            email     city       course_name  \
0         101  Alice  alice@email.com   Mumbai            Python   
1         102    Bob    bob@email.com    Delhi      Data Science   
2         105   Emma   emma@email.com  Unknown  Machine Learning   

  enrollment_date  
0      2024-01-15  
1      2024-01-20  
2      2024-02-10  

--- Analysis ---
Students in result: [101, 102, 105]
Students excluded: [104, 106, 107]
Reason: Students [104, 106, 107] don't have enrollment records in the enrollments table.


### Task 2.2: Left Join
Left join returns all rows from students table, with matching rows from enrollments.

In [5]:
# Perform Left Join: all from left table, matching from right
left_join = pd.merge(students_clean, enrollments_df, on='student_id', how='left')

print("=== LEFT JOIN RESULT ===\n")
print(f"Total rows: {len(left_join)}")
print(left_join)

# Analysis
print("\n--- Analysis ---")
null_course = left_join[left_join['course_name'].isnull()]['student_id'].unique().tolist()
print(f"Students with null course_name: {null_course}")
print(f"Reason: These students are in the students table but have no enrollment records.")

=== LEFT JOIN RESULT ===

Total rows: 6
   student_id   name            email     city       course_name  \
0         101  Alice  alice@email.com   Mumbai            Python   
1         102    Bob    bob@email.com    Delhi      Data Science   
2         104  David              NaN   Mumbai               NaN   
3         105   Emma   emma@email.com  Unknown  Machine Learning   
4         106  Frank  frank@email.com  Chennai               NaN   
5         107  Grace  grace@email.com    Delhi               NaN   

  enrollment_date  
0      2024-01-15  
1      2024-01-20  
2             NaN  
3      2024-02-10  
4             NaN  
5             NaN  

--- Analysis ---
Students with null course_name: [104, 106, 107]
Reason: These students are in the students table but have no enrollment records.


### Task 2.3: Right Join
Right join returns all rows from enrollments table, with matching rows from students.

In [6]:
# Perform Right Join: matching from left, all from right
right_join = pd.merge(students_clean, enrollments_df, on='student_id', how='right')

print("=== RIGHT JOIN RESULT ===\n")
print(f"Total rows: {len(right_join)}")
print(right_join)

# Analysis
print("\n--- Analysis ---")
null_name = right_join[right_join['name'].isnull()]['student_id'].unique().tolist()
print(f"Student_ids with null name: {null_name}")
print(f"Reason: These students have enrollment records but are not in the students table (new enrollments)")

=== RIGHT JOIN RESULT ===

Total rows: 6
   student_id   name            email     city       course_name  \
0         101  Alice  alice@email.com   Mumbai            Python   
1         102    Bob    bob@email.com    Delhi      Data Science   
2         103    NaN              NaN      NaN            Python   
3         105   Emma   emma@email.com  Unknown  Machine Learning   
4         108    NaN              NaN      NaN                AI   
5         109    NaN              NaN      NaN            Python   

  enrollment_date  
0      2024-01-15  
1      2024-01-20  
2      2024-02-01  
3      2024-02-10  
4      2024-02-15  
5      2024-03-01  

--- Analysis ---
Student_ids with null name: [103, 108, 109]
Reason: These students have enrollment records but are not in the students table (new enrollments)


### Task 2.4: Full Outer Join
Full outer join returns all rows from both tables.

In [7]:
# Perform Full Outer Join: all rows from both tables
outer_join = pd.merge(students_clean, enrollments_df, on='student_id', how='outer')

print("=== FULL OUTER JOIN RESULT ===\n")
print(f"Total rows: {len(outer_join)}")
print(outer_join)

# Analysis
print("\n--- Analysis ---")
null_name_or_course = outer_join[(outer_join['name'].isnull()) | (outer_join['course_name'].isnull())]
print(f"Rows where name is null OR course_name is null:")
print(null_name_or_course)

=== FULL OUTER JOIN RESULT ===

Total rows: 9
   student_id   name            email     city       course_name  \
0         101  Alice  alice@email.com   Mumbai            Python   
1         102    Bob    bob@email.com    Delhi      Data Science   
2         103    NaN              NaN      NaN            Python   
3         104  David              NaN   Mumbai               NaN   
4         105   Emma   emma@email.com  Unknown  Machine Learning   
5         106  Frank  frank@email.com  Chennai               NaN   
6         107  Grace  grace@email.com    Delhi               NaN   
7         108    NaN              NaN      NaN                AI   
8         109    NaN              NaN      NaN            Python   

  enrollment_date  
0      2024-01-15  
1      2024-01-20  
2      2024-02-01  
3             NaN  
4      2024-02-10  
5             NaN  
6             NaN  
7      2024-02-15  
8      2024-03-01  

--- Analysis ---
Rows where name is null OR course_name is null:
   stud

### Task 2.5: Outer Join with Indicator Column

In [8]:
# Perform Full Outer Join with indicator column
outer_join_indicator = pd.merge(students_clean, enrollments_df, on='student_id', how='outer', indicator=True)

print("=== OUTER JOIN WITH INDICATOR ===\n")
print(outer_join_indicator)

print("\n--- Merge Distribution ---")
merge_distribution = outer_join_indicator['_merge'].value_counts()
print(merge_distribution)
print("\nInterpretation:")
print(f"- left_only: {merge_distribution.get('left_only', 0)} (students with no enrollments)")
print(f"- right_only: {merge_distribution.get('right_only', 0)} (enrollments with no matching student)")
print(f"- both: {merge_distribution.get('both', 0)} (students with enrollments)")

=== OUTER JOIN WITH INDICATOR ===

   student_id   name            email     city       course_name  \
0         101  Alice  alice@email.com   Mumbai            Python   
1         102    Bob    bob@email.com    Delhi      Data Science   
2         103    NaN              NaN      NaN            Python   
3         104  David              NaN   Mumbai               NaN   
4         105   Emma   emma@email.com  Unknown  Machine Learning   
5         106  Frank  frank@email.com  Chennai               NaN   
6         107  Grace  grace@email.com    Delhi               NaN   
7         108    NaN              NaN      NaN                AI   
8         109    NaN              NaN      NaN            Python   

  enrollment_date      _merge  
0      2024-01-15        both  
1      2024-01-20        both  
2      2024-02-01  right_only  
3             NaN   left_only  
4      2024-02-10        both  
5             NaN   left_only  
6             NaN   left_only  
7      2024-02-15  right_onl

## Task 3: Lookup Operation and Automation (30 points)

Implement lookup operations and create an automation function.

### Task 3.1: Lookup Operation using .map()

In [9]:
# Create a dictionary mapping student_id to exam_score
score_dict = dict(zip(scores_df['student_id'], scores_df['exam_score']))
print("Score Dictionary:")
print(score_dict)

# Use .map() to add exam scores to students_clean
students_with_scores = students_clean.copy()
students_with_scores['exam_score'] = students_with_scores['student_id'].map(score_dict)

print("\n=== LOOKUP RESULT ===\n")
print(students_with_scores)
print("\nNote: NaN indicates students without exam scores in the scores table.")

Score Dictionary:
{101: 85, 102: 92, 104: 78, 105: 88, 106: 95}

=== LOOKUP RESULT ===

   student_id   name            email     city  exam_score
0         101  Alice  alice@email.com   Mumbai        85.0
1         102    Bob    bob@email.com    Delhi        92.0
2         104  David              NaN   Mumbai        78.0
3         105   Emma   emma@email.com  Unknown        88.0
4         106  Frank  frank@email.com  Chennai        95.0
5         107  Grace  grace@email.com    Delhi         NaN

Note: NaN indicates students without exam scores in the scores table.


### Task 3.2: Performance Comparison - Lookup vs Merge

In [10]:
# Method 2: Using pd.merge() with left join
students_merge_scores = pd.merge(students_clean, scores_df, on='student_id', how='left')

print("=== MERGE METHOD RESULT ===\n")
print(students_merge_scores)

print("\n--- Efficiency Comparison ---")
print("Lookup (.map()) vs Merge (pd.merge()):")
print("✓ .map() is MORE EFFICIENT because:")
print("  1. Creates a simple dictionary lookup (O(1) operation)")
print("  2. Minimal memory footprint - only a dict, no intermediate dataframes")
print("  3. Faster for scalar lookups (single column mapping)")
print("  4. Better for simple 1-to-1 relationships")
print("\n✓ .merge() is useful when:")
print("  1. You need to join multiple columns simultaneously")
print("  2. You need multiple columns from the lookup table")
print("  3. You need to handle many-to-many relationships")
print("\n✓ In this scenario, .map() is preferred because we only need exam_score")

=== MERGE METHOD RESULT ===

   student_id   name            email     city  exam_score
0         101  Alice  alice@email.com   Mumbai        85.0
1         102    Bob    bob@email.com    Delhi        92.0
2         104  David              NaN   Mumbai        78.0
3         105   Emma   emma@email.com  Unknown        88.0
4         106  Frank  frank@email.com  Chennai        95.0
5         107  Grace  grace@email.com    Delhi         NaN

--- Efficiency Comparison ---
Lookup (.map()) vs Merge (pd.merge()):
✓ .map() is MORE EFFICIENT because:
  1. Creates a simple dictionary lookup (O(1) operation)
  2. Minimal memory footprint - only a dict, no intermediate dataframes
  3. Faster for scalar lookups (single column mapping)
  4. Better for simple 1-to-1 relationships

✓ .merge() is useful when:
  1. You need to join multiple columns simultaneously
  2. You need multiple columns from the lookup table
  3. You need to handle many-to-many relationships

✓ In this scenario, .map() is preferr

### Task 3.3: Automation Function

In [11]:
# Create an automation function for merging
def auto_merge(df1, df2, join_type, key_column):
    """
    Automatically merge two DataFrames based on specified join type.
    
    Parameters:
    -----------
    df1 : pd.DataFrame
        First DataFrame to merge
    df2 : pd.DataFrame
        Second DataFrame to merge
    join_type : str
        Type of join: 'inner', 'left', 'right', 'outer'
    key_column : str
        Column name to merge on
    
    Returns:
    --------
    dict : Dictionary containing merged result and metadata
    """
    
    # Perform the merge
    result_df = pd.merge(df1, df2, on=key_column, how=join_type)
    
    # Return result with metadata
    return {
        'result_df': result_df,
        'row_count': len(result_df),
        'join_type': join_type,
        'columns': list(result_df.columns)
    }

print("=== AUTOMATION FUNCTION TEST ===\n")

# Test 1: Inner Join
print("Test 1: Inner Join (Students + Enrollments)")
result1 = auto_merge(students_clean, enrollments_df, 'inner', 'student_id')
print(f"Join Type: {result1['join_type']}")
print(f"Rows in Result: {result1['row_count']}")
print(f"Columns: {result1['columns']}")
print("\nResult Preview:")
print(result1['result_df'].head())

# Test 2: Left Join
print("\n" + "="*60)
print("\nTest 2: Left Join (Students + Scores)")
result2 = auto_merge(students_clean, scores_df, 'left', 'student_id')
print(f"Join Type: {result2['join_type']}")
print(f"Rows in Result: {result2['row_count']}")
print(f"Columns: {result2['columns']}")
print("\nResult Preview:")
print(result2['result_df'].head())

# Test 3: Outer Join
print("\n" + "="*60)
print("\nTest 3: Outer Join (Students + Enrollments)")
result3 = auto_merge(students_clean, enrollments_df, 'outer', 'student_id')
print(f"Join Type: {result3['join_type']}")
print(f"Rows in Result: {result3['row_count']}")
print(f"Columns: {result3['columns']}")
print("\nResult Preview:")
print(result3['result_df'])

=== AUTOMATION FUNCTION TEST ===

Test 1: Inner Join (Students + Enrollments)
Join Type: inner
Rows in Result: 3
Columns: ['student_id', 'name', 'email', 'city', 'course_name', 'enrollment_date']

Result Preview:
   student_id   name            email     city       course_name  \
0         101  Alice  alice@email.com   Mumbai            Python   
1         102    Bob    bob@email.com    Delhi      Data Science   
2         105   Emma   emma@email.com  Unknown  Machine Learning   

  enrollment_date  
0      2024-01-15  
1      2024-01-20  
2      2024-02-10  


Test 2: Left Join (Students + Scores)
Join Type: left
Rows in Result: 6
Columns: ['student_id', 'name', 'email', 'city', 'exam_score']

Result Preview:
   student_id   name            email     city  exam_score
0         101  Alice  alice@email.com   Mumbai        85.0
1         102    Bob    bob@email.com    Delhi        92.0
2         104  David              NaN   Mumbai        78.0
3         105   Emma   emma@email.com  Unkno